In [1]:
import os
from PIL import Image
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, models
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

2026-09-07 04:33:20.995710: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-07 04:33:21.054159: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2 AVX AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
dataset_path="../dataset"

In [3]:
IMG_SIZE = (384, 384)
BATCH_SIZE = 16
SEED = 42

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 5000 files belonging to 5 classes.
Using 4000 files for training.


In [5]:
val_ds = tf.keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

Found 5000 files belonging to 5 classes.
Using 1000 files for validation.


In [6]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def preprocess_resnet(image, label):
    image = preprocess_input(tf.cast(image, tf.float32))
    return image, label

train_ds = train_ds.map(
    preprocess_resnet,
    num_parallel_calls=tf.data.AUTOTUNE
)

val_ds = val_ds.map(
    preprocess_resnet,
    num_parallel_calls=tf.data.AUTOTUNE
)

train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [7]:
class_names = train_ds.class_names

print(class_names)

AttributeError: '_PrefetchDataset' object has no attribute 'class_names'

In [8]:
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(384, 384, 3)
)

In [11]:
base_model.trainable = False

print("Base model frozen.")

Base model frozen.


In [12]:
model = models.Sequential([
    base_model,

    layers.GlobalAveragePooling2D(),

    layers.Dropout(0.3),

    layers.Dense(256, activation="relu"),

    layers.Dropout(0.2),

    layers.Dense(5, activation="softmax")
])

In [13]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 12, 12, 2048)   │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 24,113,541 (91.99 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

In [15]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [16]:
checkpoint = ModelCheckpoint(
    "../models/resnet50_best.keras",

    monitor="val_accuracy",

    save_best_only=True
)

In [17]:
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",

    factor=0.2,

    patience=3,

    min_lr=1e-6
)

In [18]:
os.makedirs("../models", exist_ok=True)

In [21]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=1,
    callbacks=[
        early_stop,
        checkpoint,
        reduce_lr
    ]
)

157/157 ━━━━━━━━━━━━━━━━━━━━ 1145s 7s/step - accuracy: 0.4934 - loss: 3.1328 - val_accuracy: 0.2130 - val_loss: 1.6089 - learning_rate: 0.0010


In [ ]:
model.save("../models/resnet50_final.keras")

In [ ]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,165,201 (96.00 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 1,051,660 (4.01 MB)

In [38]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

# -------------------------------
# Load Trained Model
# -------------------------------
MODEL_PATH = "../models/resnet50_final.keras"

model = load_model(MODEL_PATH)

print("✅ Model loaded successfully!")

# -------------------------------
# Class Names
# -------------------------------
class_names = [
    "basophil",
    "erythroblast",
    "monocyte",
    "myeloblast",
    "seg_neutrophil"
]

# -------------------------------
# Image Size
# -------------------------------
IMG_SIZE = (224, 224)

# -------------------------------
# Prediction Function
# -------------------------------
def predict_image(img_path):
    img = tf.keras.utils.load_img(img_path, target_size=(224,224))

    img = tf.keras.utils.img_to_array(img)

    img = tf.expand_dims(img, 0)

    # IMPORTANT:
    # Since your training did NOT apply preprocess_input,
    # don't divide by 255 either.
    # Feed the image exactly as image_dataset_from_directory did.

    pred = model.predict(img, verbose=0)

    idx = np.argmax(pred)

    print("Prediction:", class_names[idx])

    for c, p in zip(class_names, pred[0]):
        print(f"{c:15s}: {p*100:.2f}%")

IMAGE_PATH = "../images/BA.jpg"     # <-- Change this

predict_image("../dataset/basophil/BA_580.jpg")
predict_image("../dataset/erythroblast/ERB_233.jpg")
predict_image("../dataset/monocyte/MO_1524.jpg")
predict_image("../dataset/myeloblast/MYO_0100.jpg")
predict_image("../dataset/seg_neutrophil/NGS_0022.jpg")

✅ Model loaded successfully!
Prediction: basophil
basophil       : 100.00%
erythroblast   : 0.00%
monocyte       : 0.00%
myeloblast     : 0.00%
seg_neutrophil : 0.00%
Prediction: erythroblast
basophil       : 0.00%
erythroblast   : 100.00%
monocyte       : 0.00%
myeloblast     : 0.00%
seg_neutrophil : 0.00%
Prediction: basophil
basophil       : 87.90%
erythroblast   : 0.11%
monocyte       : 11.99%
myeloblast     : 0.00%
seg_neutrophil : 0.00%
Prediction: myeloblast
basophil       : 0.01%
erythroblast   : 0.02%
monocyte       : 0.05%
myeloblast     : 99.91%
seg_neutrophil : 0.00%
Prediction: seg_neutrophil
basophil       : 0.00%
erythroblast   : 0.00%
monocyte       : 0.00%
myeloblast     : 0.00%
seg_neutrophil : 100.00%


In [33]:
import os

dataset = "../dataset"

for cls in sorted(os.listdir(dataset)):
    path = os.path.join(dataset, cls)
    if os.path.isdir(path):
        print(cls, len(os.listdir(path)))

basophil 1000
erythroblast 1000
monocyte 1000
myeloblast 1000
seg_neutrophil 1000


In [34]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 5)              │         1,285 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 25,165,201 (96.00 MB)

 Trainable params: 525,829 (2.01 MB)

 Non-trainable params: 23,587,712 (89.98 MB)

 Optimizer params: 1,051,660 (4.01 MB)

In [35]:
print(model.output_shape)

(None, 5)


In [36]:
loss, acc = model.evaluate(val_ds)

print("Accuracy:", acc)

32/32 ━━━━━━━━━━━━━━━━━━━━ 75s 2s/step - accuracy: 0.9820 - loss: 0.0528
Accuracy: 0.9819999933242798


In [37]:
images, labels = next(iter(val_ds))

predictions = model.predict(images)

predicted = np.argmax(predictions, axis=1)

for i in range(10):
    print(
        f"Actual: {class_names[labels[i].numpy()]}, "
        f"Predicted: {class_names[predicted[i]]}"
    )

1/1 ━━━━━━━━━━━━━━━━━━━━ 7s 7s/step
Actual: seg_neutrophil, Predicted: seg_neutrophil
Actual: monocyte, Predicted: monocyte
Actual: monocyte, Predicted: monocyte
Actual: basophil, Predicted: basophil
Actual: myeloblast, Predicted: myeloblast
Actual: myeloblast, Predicted: myeloblast
Actual: basophil, Predicted: basophil
Actual: basophil, Predicted: basophil
Actual: myeloblast, Predicted: myeloblast
Actual: monocyte, Predicted: monocyte
